# Ep. 03 — Memória de conversa, janela de contexto e janela deslizante

Notebook técnico: quatro progressões. Cada request é independente a menos que **você** envie o histórico.


## Setup

Shared helper: real API if `OPENAI_API_KEY` is set, otherwise offline mock (including context length exceeded).


In [ ]:
import os
import json
from typing import Any

# --- Vanilla chat helper: real OpenAI API if OPENAI_API_KEY is set, else offline mock ---
API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
USE_REAL = bool(API_KEY)

# Simulated tiny context window for demos when mocking (characters ≈ coarse "tokens")
MOCK_CONTEXT_LIMIT_CHARS = 800


def _approx_tokens(messages: list[dict]) -> int:
    return max(1, sum(len(m.get("content", "")) for m in messages) // 4)


def chat(messages: list[dict], *, max_tokens: int = 200) -> str:
    """Send chat messages. Raises RuntimeError with 'context length exceeded' when over budget."""
    if USE_REAL:
        import urllib.request
        body = json.dumps({
            "model": MODEL,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": 0,
        }).encode()
        req = urllib.request.Request(
            "https://api.openai.com/v1/chat/completions",
            data=body,
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = json.loads(resp.read().decode())
            return data["choices"][0]["message"]["content"]
        except Exception as e:
            text = str(e)
            # Surface provider context errors clearly
            if "context_length" in text.lower() or "maximum context" in text.lower() or "context length" in text.lower():
                raise RuntimeError("context length exceeded: " + text) from e
            # Try to read HTTP error body
            if hasattr(e, "read"):
                try:
                    err_body = e.read().decode()
                    if "context" in err_body.lower() and "length" in err_body.lower():
                        raise RuntimeError("context length exceeded: " + err_body) from e
                except RuntimeError:
                    raise
                except Exception:
                    pass
            raise
    # Offline mock
    total_chars = sum(len(m.get("content", "")) for m in messages)
    if total_chars > MOCK_CONTEXT_LIMIT_CHARS:
        raise RuntimeError(
            f"context length exceeded: mock limit {MOCK_CONTEXT_LIMIT_CHARS} chars, got {total_chars}"
        )
    # Find last user message and any earlier user facts
    users = [m["content"] for m in messages if m.get("role") == "user"]
    if not users:
        return "(no user message)"
    last = users[-1]
    # If history includes a prior fact about a secret/codeword, recall it
    history_text = "\n".join(users[:-1])
    if "lembra" in last.lower() or "remember" in last.lower() or "recuerd" in last.lower() or "qual" in last.lower() or "what" in last.lower() or "cuál" in last.lower() or "cual" in last.lower():
        # Look for CODEWORD / palavra / palabra patterns in earlier turns
        for u in users[:-1]:
            for marker in ("CODEWORD:", "palavra-chave:", "palabra clave:", "secret:", "segredo:", "secreto:"):
                if marker.lower() in u.lower():
                    # extract after marker
                    idx = u.lower().index(marker.lower())
                    val = u[idx + len(marker):].strip().split()[0]
                    return f"Recalled from history: {val}"
        if len(users) == 1:
            return "I have no prior messages in this request, so I don't know."
        return "I don't see that fact in the provided history."
    if len(users) == 1:
        return f"Got it. (stateless ack) Stored nothing server-side."
    return f"OK. History turns={len(users)}. Last: {last[:80]}"


def sliding_window(messages: list[dict], max_messages: int) -> list[dict]:
    """Keep system (if any) + the most recent messages up to max_messages total dialog slots."""
    if not messages:
        return []
    system = [m for m in messages if m.get("role") == "system"]
    dialog = [m for m in messages if m.get("role") != "system"]
    keep_dialog = max(0, max_messages - len(system))
    trimmed = dialog[-keep_dialog:] if keep_dialog else []
    return system + trimmed


print("USE_REAL_API =", USE_REAL, "| MODEL =", MODEL if USE_REAL else "(mock)")
print("MOCK_CONTEXT_LIMIT_CHARS =", MOCK_CONTEXT_LIMIT_CHARS)


## 1) O LLM esquece

Duas chamadas **separadas**. A segunda não leva histórico, então o modelo não lembra o fato da primeira.


In [ ]:
fato = "CODEWORD: ORBIT-77"
r1 = chat([{"role": "user", "content": f"Lembre deste segredo: {fato}"}])
print("Chamada 1:", r1)

r2 = chat([{"role": "user", "content": "Qual é o CODEWORD que eu te disse?"}])
print("Chamada 2 (sem histórico):", r2)
print("→ Esperado: NÃO sabe ORBIT-77.")


## 2) Montar o histórico

Acumule mensagens numa lista (antigo → recente). Reenvie a lista a cada turno para o modelo "lembrar".


In [ ]:
historico: list[dict] = []
historico.append({"role": "user", "content": "Lembre deste segredo: CODEWORD: ORBIT-77"})
a1 = chat(historico)
historico.append({"role": "assistant", "content": a1})
print("Após turno 1, len =", len(historico))

historico.append({"role": "user", "content": "Qual é o CODEWORD que eu te disse?"})
a2 = chat(historico)
historico.append({"role": "assistant", "content": a2})
print("Com histórico:", a2)
print("→ Esperado: recupera ORBIT-77 da lista que você enviou.")


## 3) Estourar a janela de contexto

Continue adicionando até o provedor devolver **context length exceeded** (erro real se houver chave; senão mock fiel).


In [ ]:
big = list(historico)
blob = "PADDING " * 50
errored = False
for i in range(1, 40):
    big.append({"role": "user", "content": f"turno-{i}: {blob}"})
    try:
        _ = chat(big, max_tokens=16)
        print(f"ok crescimento {i}, messages={len(big)}")
    except RuntimeError as e:
        print("ERRO:", e)
        errored = True
        break
print("Recebeu context length exceeded =", errored)


## 4) Janela deslizante

Quando o orçamento enche, drope as mensagens de diálogo **mais antigas** (preserve `system`). O modelo esquece o começo, mantém o fio recente — e para de falhar.


In [ ]:
msgs = [{"role": "system", "content": "Você é um assistente conciso."}]
msgs.append({"role": "user", "content": "Fato inicial CODEWORD: ORBIT-77 — dito só no começo."})
msgs.append({"role": "assistant", "content": "Ok, fato inicial."})
for i in range(20):
    msgs.append({"role": "user", "content": f"Assunto recente {i}: " + ("x" * 40)})
    msgs.append({"role": "assistant", "content": f"Ack {i}"})

print("Antes do trim: messages =", len(msgs), "approx_tokens ≈", _approx_tokens(msgs))
try:
    chat(msgs, max_tokens=16)
    print("Inesperado: chamada grande passou")
except RuntimeError as e:
    print("Erro antes do trim (esperado):", e)

trimmed = sliding_window(msgs, max_messages=8)
print("Após sliding_window(max_messages=8):", len(trimmed), "messages")
for m in trimmed:
    print(" ", m["role"], ":", m["content"][:60].replace("\n", " "), "...")

out = chat(trimmed + [{"role": "user", "content": "Qual era o CODEWORD do começo?"}])
print("Pergunta sobre o começo após trim:", out)
print("→ CODEWORD inicial tende a sumir; fio recente ainda cabe.")
out2 = chat(trimmed + [{"role": "user", "content": "Qual foi o último índice de assunto recente?"}])
print("Pergunta sobre o recente:", out2)
